# Data quality and label review

Run the pipeline first. This notebook reads the database without writing. Unknown outcomes are not healthy negatives. See the README and generated reports for the study definition.

In [1]:
from pathlib import Path
import json
import sys
import duckdb

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
con = duckdb.connect(str(ROOT / 'database/mlb_ucl.duckdb'), read_only=True)
report = json.loads((ROOT / 'reports/validation_report.json').read_text())
print('Validation passed:', report['passed'])
print('Rows:', report['row_counts']['pitches'], 'pitches;', report['row_counts']['appearances'], 'appearances')

Validation passed: True
Rows: 133403 pitches; 3826 appearances


In [2]:
print(con.sql('SELECT is_starter, count(*) AS rows, count(*) FILTER (WHERE ucl_injury_next_30d) AS positive_rows, count(DISTINCT CASE WHEN ucl_injury_next_30d THEN pitcher_id END) AS positive_pitchers FROM training_pitcher_appearances GROUP BY is_starter').df().to_string(index=False))

 is_starter  rows  positive_rows  positive_pitchers
       True   825             11                  5
      False  2648             12                  2


## Missingness

Do not fill unknown labels with zero. Fit measurement imputation and any feature selection only within training folds.

In [3]:
import pandas as pd
missing = pd.read_csv(ROOT / 'reports/model_missingness.csv')
print(missing.sort_values('missing_fraction', ascending=False).head(20).to_string(index=False))

                variable  missing_rows  missing_fraction
sv_release_z_avg_prev_10          3826          1.000000
sv_release_x_avg_prev_10          3826          1.000000
     sv_spin_avg_prev_10          3826          1.000000
 sv_avg_velocity_prev_10          3826          1.000000
sv_extension_avg_prev_10          3826          1.000000
 sv_release_x_avg_prev_5          3810          0.995818
 sv_release_z_avg_prev_5          3810          0.995818
 sv_extension_avg_prev_5          3810          0.995818
  sv_avg_velocity_prev_5          3810          0.995818
      sv_spin_avg_prev_5          3810          0.995818
kc_release_z_avg_prev_10          3801          0.993466
 kc_avg_velocity_prev_10          3801          0.993466
     kc_spin_avg_prev_10          3801          0.993466
kc_release_x_avg_prev_10          3801          0.993466
kc_extension_avg_prev_10          3801          0.993466
  sv_avg_velocity_prev_3          3799          0.992943
 sv_extension_avg_prev_3       

In [4]:
print(con.sql('SELECT pitcher_name, game_date, next_known_ucl_onset, label_onset_basis FROM training_pitcher_appearances WHERE ucl_injury_next_30d ORDER BY pitcher_id, game_date').df().to_string(index=False))

   pitcher_name  game_date next_known_ucl_onset                        label_onset_basis
     Wade Miley 2024-04-10           2024-04-22 elbow_il_report_linked_by_mlbam_and_time
     Wade Miley 2024-04-16           2024-04-22 elbow_il_report_linked_by_mlbam_and_time
   Brooks Raley 2024-03-31           2024-04-21 elbow_il_report_linked_by_mlbam_and_time
   Brooks Raley 2024-04-01           2024-04-21 elbow_il_report_linked_by_mlbam_and_time
   Brooks Raley 2024-04-04           2024-04-21 elbow_il_report_linked_by_mlbam_and_time
   Brooks Raley 2024-04-05           2024-04-21 elbow_il_report_linked_by_mlbam_and_time
   Brooks Raley 2024-04-07           2024-04-21 elbow_il_report_linked_by_mlbam_and_time
   Brooks Raley 2024-04-14           2024-04-21 elbow_il_report_linked_by_mlbam_and_time
   Brooks Raley 2024-04-15           2024-04-21 elbow_il_report_linked_by_mlbam_and_time
   Brooks Raley 2024-04-19           2024-04-21 elbow_il_report_linked_by_mlbam_and_time
   Tejay Antone 2024-

In [5]:
print(con.sql('SELECT player_name, surgery_date_raw, surgery_date_precision, known_onset_date, il_placement_date, identity_method FROM injuries WHERE known_onset_date IS NOT NULL ORDER BY known_onset_date').df().to_string(index=False))

     player_name surgery_date_raw surgery_date_precision known_onset_date il_placement_date                        identity_method
   Lucas Giolito        3/12/2024                    day       2024-01-03        2024-03-24 unique_exact_normalized_name_crosswalk
 Jesse Scholtens         3/1/2024   month_or_exact_day_1       2024-02-27        2024-02-27                           mlbam_source
   Jackson Kowar         3/1/2024   month_or_exact_day_1       2024-03-10        2024-03-10                           mlbam_source
    Dauri Moreta         3/1/2024   month_or_exact_day_1       2024-03-16        2024-03-16                           mlbam_source
     Luis Patino        4/30/2024                    day       2024-03-19        2024-03-19                           mlbam_source
      Eury Perez         4/8/2024                    day       2024-03-28        2024-03-25                           mlbam_source
      Matt Brash         5/8/2024                    day       2024-03-28        20

## Confirm X excludes label metadata

The allowlist contains prospective inputs only. Numeric audit columns and future outcome fields are not predictors. Use a temporal split with horizon purging; count distinct injuries, not just positive rows.

In [6]:
feature_columns = json.loads((ROOT / 'data/exports/feature_columns.json').read_text())
df = con.sql('SELECT * FROM starter_training_appearances ORDER BY game_date, pitcher_id').df()
X = df[feature_columns]
y = df['ucl_injury_next_30d'].astype(int)
assert not any('next_' in c or 'return_date' in c for c in feature_columns)
print('X:', X.shape, 'y:', y.value_counts().to_dict())
print('All-missing candidate features in this short starter sample:', X.isna().all().sum())
con.close()

X: (825, 459) y: {0: 814, 1: 11}
All-missing candidate features in this short starter sample: 30
